In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        SELEÇÃO DE VARIÁVEIS — NORDESTE                       ║
# ║       Ensemble Spearman + Mutual Information + LightGBM | Cenário Único      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : [Nome da Instituição]
#  Curso       : [Nome do Curso]
#  Data        : Junho / 2025
#  Versão      : 7.1 (Apenas Cenário A + Fix PerformanceWarning)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de seleção de variáveis (Step 9) da
#  arquitetura de dados do TCC, especificamente para o submercado NORDESTE.
#  Ele recebe os Parquets com engenharia de atributos já aplicada (etapa
#  anterior) e seleciona um conjunto final de features para o modelo de
#  previsão do PLD, combinando três métodos complementares em um ensemble:
#
#    • Correlação de Spearman (relações monotônicas, lineares e não-lineares)
#    • Mutual Information (dependência estatística geral, não só monotônica)
#    • Importância de features do LightGBM (relações nas quais o modelo
#      efetivamente se apoia, incluindo interações)
#
#  Antes do ensemble, o script aplica dois pré-filtros de redução de
#  candidatas: remoção de variáveis com variância zero e remoção de
#  variáveis altamente correlacionadas entre si (multicolinearidade), ambos
#  calculados apenas sobre uma amostra do treino.
#
#  Um conjunto de features consideradas obrigatórias (calendário, regime de
#  preço/volatilidade, indicadores de escassez etc.) é sempre incluído na
#  seleção final; o restante das vagas é preenchido pelo ranking do
#  ensemble.
#
#  Ao final da execução, o script gera:
#    (a) Parquets de treino e teste apenas com as features selecionadas;
#    (b) um CSV de ranking completo de todas as candidatas avaliadas;
#    (c) um TXT legível com a lista de features obrigatórias e por ranking;
#    (d) um JSON com todos os parâmetros e a seleção final (auditável);
#    (e) quatro figuras (PNG) de suporte à análise da seleção.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES GLOBAIS"):
#
#    SCENARIOS             : cenário(s) processado(s) (padrão: apenas
#                             cenario_A_todos_anos)
#    STRATEGIES            : estratégia(s) de engenharia de atributos
#                             herdadas da etapa anterior (padrão:
#                             HYBRID_AGGRESSIVE)
#    TARGET_CANONICAL       : nome exato esperado da coluna alvo do Nordeste
#    FEATURES_OBRIGATORIAS  : lista de features sempre incluídas na seleção
#    PIPELINE_CONFIG        : hiperparâmetros do pipeline (nº de features
#                             finais, limiares de multicolinearidade, tamanho
#                             de amostra, pesos do ensemble, parâmetros do
#                             LightGBM, limiares de alerta de RAM etc.)
#    BASE_DIR               : caminho raiz de armazenamento
#                             (Google Drive: /content/drive/MyDrive/TCC_PLD_Project)
#
#  Arquivos de entrada esperados (gerados pela etapa anterior de engenharia
#  de atributos — Step 8):
#    .../8_engenharia_atributos/<cenario>/treino/<ESTRATEGIA>/TREINO_FEATURES.parquet
#    .../8_engenharia_atributos/<cenario>/teste/<ESTRATEGIA>/TESTE_FEATURES.parquet
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Parquets selecionados →  .../9_selecao_variaveis/<cenario>/<ESTRATEGIA>/
#     TREINO_SELECTED.parquet
#     TESTE_SELECTED.parquet
#     (apenas as colunas selecionadas + coluna target, em float32)
#
#  2. Ranking completo (CSV) →  mesma pasta / RANKING_<ESTRATEGIA>.csv
#     Uma linha por candidata avaliada, com posição no ranking, tipo
#     (obrigatória/ranking), se foi selecionada, score do ensemble e os
#     três scores individuais (Spearman, MI, LightGBM).
#
#  3. Lista legível (TXT)   →  mesma pasta / FEATURES_<ESTRATEGIA>.txt
#     Cabeçalho com metadados da execução, lista de obrigatórias e lista
#     das features selecionadas por ranking, com seus scores.
#
#  4. Parâmetros (JSON)     →  mesma pasta / feature_params_selection.json
#     Pesos do ensemble, contagens em cada etapa do funil de seleção,
#     configuração completa do pipeline e a lista final de features —
#     documento auditável da seleção realizada.
#
#  5. Figuras (PNG)         →  mesma pasta / "figuras" /
#     ensemble_score_top<N>_<ESTRATEGIA>.png   — ranking visual das top-N
#     metodos_comparacao_<ESTRATEGIA>.png      — dispersão dos 3 métodos vs. ensemble
#     spearman_heatmap_<ESTRATEGIA>.png        — heatmap de correlação com o target
#     ensemble_distribution_<ESTRATEGIA>.png   — distribuição do score ensemble
#
#  6. Saída no terminal (Rich):
#     • Painéis de arquitetura anti-leakage e progresso por método
#     • Tabela de resumo do funil de seleção (candidatas → filtros → final)
#     • Tabela de ranking completo com scores por método
#     • Relatório de tempo por estágio e tabela consolidada final
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ARQUITETURA ANTI-LEAKAGE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Variância zero, multicolinearidade, Spearman, Mutual Information e
#  LightGBM são calculados SOMENTE sobre uma amostra do TREINO. O TESTE
#  nunca influencia nenhuma decisão de seleção; apenas as colunas já
#  selecionadas são recortadas do conjunto de teste ao final.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_DIR.
#     Depende dos Parquets de treino/teste com engenharia de atributos já
#     aplicada (Step 8), para o mesmo cenário e estratégia.
#
#  2. DETECÇÃO DINÂMICA DO TARGET
#     detectar_target() primeiro procura o nome exato esperado
#     (TARGET_CANONICAL); se não encontrar, cai para uma busca por tokens
#     ("nordeste" + "target") e, por fim, para combinações de fallback.
#     Isso torna o script tolerante a pequenas variações de nomenclatura
#     entre execuções do pipeline anterior, sem arriscar selecionar a
#     coluna errada silenciosamente — todos os fallbacks são registrados no
#     terminal.
#
#  3. EXCLUSÃO DE TARGETS DE OUTROS SUBMERCADOS
#     EXCLUDE_PATTERNS lista os nomes (ou fragmentos) das colunas-alvo dos
#     demais submercados (Sul, Norte, Sudeste). Essas colunas são sempre
#     excluídas do conjunto de candidatas, mesmo que estejam presentes no
#     Parquet, para evitar vazamento entre os targets dos diferentes
#     modelos regionais.
#
#  4. FEATURES OBRIGATÓRIAS
#     Um conjunto fixo de variáveis de calendário, regime de preço/
#     volatilidade e indicadores de pressão do sistema é sempre incluído na
#     seleção final, independentemente do ranking do ensemble — essas
#     variáveis carregam conhecimento de domínio que não necessariamente
#     aparece nos topos dos métodos estatísticos isolados.
#
#  5. PRÉ-FILTROS ANTES DO ENSEMBLE
#     _remove_zero_variance() descarta candidatas sem variabilidade na
#     amostra do treino. _remove_multicolinearidade() usa ranks
#     normalizados e produto matricial vetorizado para descartar
#     candidatas fortemente correlacionadas entre si, mantendo apenas uma
#     representante de cada grupo altamente colinear (limiar configurável
#     em `multicolinearidade_thresh`).
#
#  6. TRÊS MÉTODOS DE PONTUAÇÃO + ENSEMBLE POR RANK
#     Cada método (_compute_spearman, _compute_mi, _compute_lgbm) processa
#     as candidatas em chunks para controlar o uso de memória. O
#     _compute_ensemble_score() converte cada score para um rank
#     normalizado em [0, 1] antes de combiná-los — isso evita que a escala
#     natural de cada método (correlação, informação mútua, ganho do
#     LightGBM) distorça o peso relativo na combinação final.
#
#  7. POOL REDUZIDO PARA O LIGHTGBM
#     Para conter o custo computacional, o LightGBM não é treinado sobre
#     todas as candidatas remanescentes, e sim sobre a união das
#     obrigatórias com as top-N candidatas de Spearman e de Mutual
#     Information (`top_corr_candidates`), o que já cobre a grande maioria
#     das variáveis relevantes por ambos os critérios.
#
#  8. MONITORAMENTO DE RAM
#     _print_ram() reporta o uso de memória em pontos-chave do pipeline e
#     aciona um gc.collect() preventivo quando o uso ultrapassa o limiar
#     crítico configurado (`ram_crit_pct`), reduzindo o risco de falha por
#     estouro de memória em ambientes com RAM limitada (ex.: Colab free).
#
#  9. SUPRESSÃO DE AVISO DE PERFORMANCE (PerformanceWarning)
#     pandas.errors.PerformanceWarning é silenciado explicitamente após o
#     import do pandas. Esse aviso é disparado por
#     _remove_multicolinearidade() ao inserir blocos de colunas no
#     DataFrame de ranks (fragmentação interna do DataFrame) — é apenas um
#     aviso de performance, não um erro, e não afeta o resultado da
#     seleção.
#
#  10. REPRODUTIBILIDADE
#      Todas as etapas usam `random_state` fixo (42) para amostragem e
#      seeds fixas e incrementais para os múltiplos treinos do LightGBM. O
#      JSON de parâmetros registra a configuração completa do pipeline e a
#      lista final de features, permitindo auditar e reproduzir a seleção.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações vetorizadas (ranks, correlações)
#  pandas            1.5            Manipulação de DataFrames
#  scipy             1.9            rankdata (base do cálculo de Spearman)
#  scikit-learn      1.1            mutual_info_regression
#  lightgbm          3.3            Modelo de importância de features
#  matplotlib        3.6           Geração das figuras
#  psutil            5.9           Monitoramento de uso de RAM
#  rich              13.0           Painéis, tabelas e progresso no terminal
#
#  Instalação rápida (Google Colab / pip):
#      !pip install lightgbm psutil rich
#  (numpy, pandas, scipy, scikit-learn e matplotlib já estão disponíveis no
#  Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Entrada (Step 8):
#    .../8_engenharia_atributos/
#    └── cenario_A_todos_anos/
#        ├── treino/<ESTRATEGIA>/TREINO_FEATURES.parquet
#        └── teste/<ESTRATEGIA>/TESTE_FEATURES.parquet
#
#  Saída (Step 9):
#    .../9_selecao_variaveis/
#    ├── cenario_A_todos_anos/
#    │   └── <ESTRATEGIA>/
#    │       ├── TREINO_SELECTED.parquet
#    │       ├── TESTE_SELECTED.parquet
#    │       ├── RANKING_<ESTRATEGIA>.csv
#    │       ├── FEATURES_<ESTRATEGIA>.txt
#    │       ├── feature_params_selection.json
#    │       └── figuras/
#    │           ├── ensemble_score_top<N>_<ESTRATEGIA>.png
#    │           ├── metodos_comparacao_<ESTRATEGIA>.png
#    │           ├── spearman_heatmap_<ESTRATEGIA>.png
#    │           └── ensemble_distribution_<ESTRATEGIA>.png
#    └── relatorio_selecao_variaveis_nordeste.csv
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install lightgbm psutil rich
#
#  Célula 3 — Executar o script (após o Step 8 já ter sido executado):
#      exec(open('selecao_variaveis_nordeste_v71.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import gc
import json
import re
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import psutil
from lightgbm import LGBMRegressor
from rich import box
from rich.console import Console
from rich.panel import Panel
from rich.progress import BarColumn, Progress, SpinnerColumn, TextColumn, TimeElapsedColumn
from rich.rule import Rule
from rich.table import Table
from rich.text import Text
from rich.theme import Theme
from sklearn.feature_selection import mutual_info_regression

# Silencia o PerformanceWarning de DataFrame fragmentado.
# Precisa vir DEPOIS do import do pandas (pd.errors só existe após o import).
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# ==============================================================================
# 🎨 TEMA VISUAL
# ==============================================================================
THEME = Theme({
    "info"      : "bold cyan",
    "warning"   : "bold yellow",
    "error"     : "bold red",
    "success"   : "bold green",
    "header"    : "bold white on dark_blue",
    "highlight" : "bold magenta",
    "muted"     : "dim white",
    "mandatory" : "bold yellow",
    "selected"  : "bold green",
    "rejected"  : "dim red",
    "rank_top"  : "bold cyan",
    "rank_mid"  : "cyan",
    "rank_low"  : "dim cyan",
    "ram_ok"    : "bold green",
    "ram_warn"  : "bold yellow",
    "ram_crit"  : "bold red",
    "cenario_a" : "bold green",
    "cenario_b" : "bold magenta",
    "ensemble"  : "bold white",
})
console = Console(theme=THEME)

# ==============================================================================
# ⚙️  CONFIGURAÇÕES GLOBAIS
# ==============================================================================
BASE_DIR  = Path("/content/drive/MyDrive/TCC_PLD_Project/09-ESCRITA_TCC/PARTES_TCC/codigos/dados")
INPUT_DIR = BASE_DIR / "8_engenharia_atributos"
SAVE_DIR  = BASE_DIR / "9_selecao_variaveis"

# ── Cenários que o script processa ────────────────────────────────────────────
# Apenas o Cenário A (todos os anos do treino).
SCENARIOS: List[Dict] = [
    {
        "label": "cenario_A_todos_anos",
        "desc" : "FIT com todos os anos do treino",
        "cor"  : "cenario_a",
    },
]

# ── Estratégias de engenharia de atributos (vindas do passo anterior) ─────────
STRATEGIES: List[str] = [
    "HYBRID_AGGRESSIVE",
]

# ── TARGET do Nordeste ────────────────────────────────────────────────────────
TARGET_CANONICAL : str       = "01_CCEE_NORDESTE_TARGET_PLD_HORA_NORDESTE"
_TARGET_TOKENS   : List[str] = ["nordeste", "target"]

# ── Targets de OUTROS submercados — devem ser excluídos como candidatas ───────
EXCLUDE_PATTERNS: List[str] = [
    "target_pld_hora_sul",
    "target_pld_hora_norte",
    "target_pld_hora_sudeste",
    "target_preco_horario",
    "_target_pld_hora_sul",
    "_target_pld_hora_norte",
    "_target_pld_hora_sudeste",
    "01_ccee_norte_target",
    "01_ccee_sul_target",
    "01_ccee_sudeste_target",
]

# ── Features obrigatórias ─────────────────────────────────────────────────────
FEATURES_OBRIGATORIAS: List[str] = [
    "KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA",
    "VOL_ANUAL", "REGIME_VOL", "REGIME_PRECO", "VOL_MENSAL",
    "TENDENCIA_NORM",
    "DIA_SEMANA", "IS_FIM_DE_SEMANA", "IS_FERIADO", "IS_DIA_UTIL",
    "HORA", "HORA_SEN", "HORA_COS",
    "MEDIA_HORA",
    "MES_SEN", "MES_COS",
    "PRESSAO_SISTEMA",
    "EAR_ZSCORE", "ALERTA_ESCASSEZ",
    "CMO_MEDIA_SUBMERCADOS",
    "HORARIO_PICO",
]

# ── Parâmetros do pipeline ────────────────────────────────────────────────────
PIPELINE_CONFIG: Dict = {
    "n_total_features"          : 80,
    "multicolinearidade_thresh" : 0.95,
    "sub_sample_multicol"       : 6_000,
    "sample_size"               : 20_000,
    "spearman_chunk"            : 150,
    "mi_n_neighbors"            : 5,
    "mi_random_state"           : 42,
    "mi_chunk"                  : 150,
    "lgbm_n_seeds"              : 2,
    "lgbm_n_estimators"         : 200,
    "lgbm_num_leaves"           : 31,
    "lgbm_learning_rate"        : 0.05,
    "lgbm_subsample"            : 0.7,
    "lgbm_colsample_bytree"     : 0.7,
    "lgbm_min_child_samples"    : 30,
    "lgbm_n_jobs"               : 1,
    "spearman_weight"           : 0.30,
    "mi_weight"                 : 0.30,
    "lgbm_weight"               : 0.40,
    "top_corr_candidates"       : 180,
    "ram_warn_pct"              : 65,
    "ram_crit_pct"              : 80,
    "fig_top_n_importance"      : 40,
    "fig_heatmap_top_n"         : 30,
    "fig_dpi"                   : 120,
}

# ── Mapa cor -> border_style Rich válido ──────────────────────────────────────
_BORDER_STYLE_MAP: Dict[str, str] = {
    "cenario_a": "green",
    "cenario_b": "magenta",
}

# ==============================================================================
# 🎯 DETECÇÃO DINÂMICA DE TARGET
# ==============================================================================

def detectar_target(df: pd.DataFrame) -> str:
    if TARGET_CANONICAL in df.columns:
        console.print(f"   [success]✔  TARGET (nome exato): [bold]{TARGET_CANONICAL}[/][/]")
        return TARGET_CANONICAL

    cols_lower = {c: c.lower() for c in df.columns}

    candidatas = [c for c, cl in cols_lower.items()
                  if all(tok in cl for tok in _TARGET_TOKENS)]
    if candidatas:
        escolhida = candidatas[0]
        console.print(f"   [warning]⚠  TARGET por tokens {_TARGET_TOKENS}: [bold]{escolhida}[/][/]")
        return escolhida

    for tokens in [["target", "nordeste"], ["pld", "nordeste"]]:
        cands = [c for c, cl in cols_lower.items() if all(t in cl for t in tokens)]
        if cands:
            console.print(f"   [warning]⚠  TARGET via fallback {tokens}: [bold]{cands[0]}[/][/]")
            return cands[0]

    raise KeyError(
        f"Coluna TARGET não encontrada.\n"
        f"Canônico esperado: {TARGET_CANONICAL!r}\n"
        f"Colunas disponíveis (30 primeiras): {list(df.columns[:30])}"
    )


def _is_excluded(col: str) -> bool:
    cl = col.lower()
    return any(pat in cl for pat in EXCLUDE_PATTERNS)


# ==============================================================================
# ⏱️  StageTimer
# ==============================================================================
@dataclass
class StageTimer:
    records: Dict[str, float] = field(default_factory=dict)
    _start : float            = field(default=0.0, init=False, repr=False)

    @contextmanager
    def track(self, label: str):
        self._start = time.perf_counter()
        try:
            yield
        finally:
            self.records[label] = round(time.perf_counter() - self._start, 2)


# ==============================================================================
# 📋 DATACLASS — Resultado por estratégia + cenário
# ==============================================================================
@dataclass
class SelectionResult:
    cenario              : str
    strategy             : str
    target_col           : str
    mandatory            : List[str]
    top_additional       : List[str]
    final_features       : List[str]
    spearman_dict        : Dict[str, float]
    mi_dict              : Dict[str, float]
    lgbm_importance      : pd.Series
    ensemble_score       : pd.Series
    n_candidates_initial : int
    n_after_variance     : int
    n_after_multicol     : int
    n_treino_rows        : int
    n_teste_rows         : int


# ==============================================================================
# 🏗️  MOTOR — FeatureSelectionEngine
# ==============================================================================
class FeatureSelectionEngine:

    def __init__(
        self,
        input_dir : Path,
        save_dir  : Path,
        scenarios : List[Dict],
        strategies: List[str],
        config    : Dict,
    ) -> None:
        self.input_dir  = Path(input_dir)
        self.save_dir   = Path(save_dir)
        self.scenarios  = scenarios
        self.strategies = strategies
        self.cfg        = config
        self.timer      = StageTimer()
        self._all_results: List[Dict] = []

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 UTILITÁRIOS DE RAM
    # ──────────────────────────────────────────────────────────────────────────

    def _ram_status(self) -> Tuple[float, float, str]:
        mem   = psutil.virtual_memory()
        used  = mem.used  / 1024**3
        tot   = mem.total / 1024**3
        pct   = mem.percent
        style = (
            "ram_crit" if pct >= self.cfg["ram_crit_pct"] else
            "ram_warn" if pct >= self.cfg["ram_warn_pct"] else
            "ram_ok"
        )
        return used, tot, style

    def _print_ram(self, label: str = "") -> None:
        used, tot, style = self._ram_status()
        pct = used / tot * 100
        tag = f" [{label}]" if label else ""
        console.print(f"   [{style}]🧠 RAM{tag}: {used:.1f}/{tot:.1f} GB ({pct:.0f}%)[/{style}]")
        if style == "ram_crit":
            console.print("   [ram_crit]⚠️  RAM CRÍTICA — gc.collect() preventivo[/ram_crit]")
            gc.collect()

    def _force_free(self, *objs) -> None:
        for o in objs:
            try:
                del o
            except Exception:
                pass
        gc.collect()

    @staticmethod
    def _clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
        df.columns = [re.sub(r'[":{},\[\]]', "_", str(c)) for c in df.columns]
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 📂 CARREGAMENTO
    # ──────────────────────────────────────────────────────────────────────────

    def _load_split(self, path: Path, split_label: str) -> pd.DataFrame:
        console.print(f"   [info]📂 Lendo {split_label}: {path.name}[/]")
        df = pd.read_parquet(path)
        df = self._clean_col_names(df)

        key_cols = [c for c in df.columns if c.startswith("KEY_")]
        num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in key_cols]

        for col in num_cols:
            df[col] = (
                df[col]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0.0)
                .astype(np.float16)
            )

        obj_cols = [c for c in df.select_dtypes(exclude=[np.number]).columns
                    if c not in key_cols]
        if obj_cols:
            df = df.drop(columns=obj_cols)
            console.print(f"   [muted]↳ {len(obj_cols)} colunas object descartadas[/]")

        mem_mb = df.memory_usage(deep=True).sum() / 1024**2
        console.print(f"   [info]📐 Shape {split_label}: {df.shape} | 💾 {mem_mb:.1f} MB[/]")
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔍 PRÉ-FILTRO 1 — Variância zero
    # ──────────────────────────────────────────────────────────────────────────

    def _remove_zero_variance(
        self, candidates: List[str], sample: pd.DataFrame
    ) -> Tuple[List[str], int]:
        stds  = sample[candidates].std()
        valid = stds[stds > 1e-5].index.tolist()
        removed = len(candidates) - len(valid)
        return valid, removed

    # ──────────────────────────────────────────────────────────────────────────
    # 🔍 PRÉ-FILTRO 2 — Multicolinearidade
    # ──────────────────────────────────────────────────────────────────────────

    def _remove_multicolinearidade(
        self, candidates: List[str], sample: pd.DataFrame
    ) -> Tuple[List[str], int]:
        thresh = self.cfg["multicolinearidade_thresh"]
        to_drop: set       = set()
        kept   : List[str] = []

        ranks = pd.DataFrame(index=sample.index, dtype=np.float32)
        for i in range(0, len(candidates), 100):
            blk = candidates[i: i + 100]
            ranks[blk] = sample[blk].rank(method="average").astype(np.float32)

        means      = ranks.mean()
        stds_r     = ranks.std().replace(0, 1e-9)
        ranks_norm = ((ranks - means) / stds_r).values
        col_idx    = {c: i for i, c in enumerate(candidates)}

        with Progress(
            SpinnerColumn(),
            TextColumn("[info]{task.description}[/info]"),
            BarColumn(bar_width=25),
            TextColumn("[muted]{task.completed}/{task.total}[/muted]"),
            console=console, transient=True,
        ) as prog:
            task = prog.add_task("Multicolinearidade...", total=len(candidates))

            for col in candidates:
                if col in to_drop:
                    prog.advance(task)
                    continue
                if not kept:
                    kept.append(col)
                    prog.advance(task)
                    continue

                col_vec  = ranks_norm[:, col_idx[col]]
                kept_mat = ranks_norm[:, [col_idx[k] for k in kept]]
                corrs    = np.abs(kept_mat.T @ col_vec) / len(col_vec)

                if corrs.max() >= thresh:
                    to_drop.add(col)
                else:
                    kept.append(col)

                prog.advance(task)

        self._force_free(ranks, ranks_norm)
        return kept, len(to_drop)

    # ──────────────────────────────────────────────────────────────────────────
    # 📐 MÉTODO A — Spearman
    # ──────────────────────────────────────────────────────────────────────────

    def _compute_spearman(
        self,
        candidates: List[str],
        sample    : pd.DataFrame,
        target    : str,
    ) -> Dict[str, float]:
        from scipy.stats import rankdata

        chunk_size = self.cfg["spearman_chunk"]
        y_rank     = rankdata(sample[target].values.astype(np.float64)).astype(np.float32)
        y_rc       = y_rank - y_rank.mean()
        y_norm     = y_rc / (np.linalg.norm(y_rc) + 1e-9)

        results: Dict[str, float] = {}

        with Progress(
            SpinnerColumn(), TextColumn("[info]Spearman...[/info]"),
            BarColumn(bar_width=20),
            TextColumn("[success]{task.completed}/{task.total}[/success]"),
            TimeElapsedColumn(), console=console, transient=True,
        ) as prog:
            task = prog.add_task("Spearman", total=len(candidates))

            for i in range(0, len(candidates), chunk_size):
                chunk  = candidates[i: i + chunk_size]
                X_mat  = sample[chunk].values.astype(np.float64)
                X_rank = np.apply_along_axis(rankdata, 0, X_mat).astype(np.float32)
                del X_mat
                X_rc   = X_rank - X_rank.mean(axis=0)
                norms  = np.linalg.norm(X_rc, axis=0) + 1e-9
                X_norm = (X_rc / norms).astype(np.float32)
                del X_rank, X_rc
                corrs  = np.abs(y_norm @ X_norm)
                results.update(zip(chunk, corrs.tolist()))
                del X_norm, corrs
                prog.advance(task, len(chunk))

        gc.collect()
        return results

    # ──────────────────────────────────────────────────────────────────────────
    # 📐 MÉTODO B — Mutual Information
    # ──────────────────────────────────────────────────────────────────────────

    def _compute_mi(
        self,
        candidates: List[str],
        sample    : pd.DataFrame,
        target    : str,
    ) -> Dict[str, float]:
        chunk_size = self.cfg["mi_chunk"]
        y          = sample[target].values.astype(np.float32)
        results    : Dict[str, float] = {}

        with Progress(
            SpinnerColumn(), TextColumn("[info]Mutual Information...[/info]"),
            BarColumn(bar_width=20),
            TextColumn("[highlight]{task.completed}/{task.total}[/highlight]"),
            TimeElapsedColumn(), console=console, transient=True,
        ) as prog:
            task = prog.add_task("MI", total=len(candidates))

            for i in range(0, len(candidates), chunk_size):
                chunk   = candidates[i: i + chunk_size]
                X_chunk = sample[chunk].values.astype(np.float32)

                mi_vals = mutual_info_regression(
                    X_chunk, y,
                    n_neighbors  = self.cfg["mi_n_neighbors"],
                    random_state = self.cfg["mi_random_state"],
                )
                results.update(zip(chunk, mi_vals.tolist()))
                del X_chunk, mi_vals
                prog.advance(task, len(chunk))

        gc.collect()
        return results

    # ──────────────────────────────────────────────────────────────────────────
    # 📐 MÉTODO C — LightGBM
    # ──────────────────────────────────────────────────────────────────────────

    def _compute_lgbm(
        self,
        features: List[str],
        sample  : pd.DataFrame,
        target  : str,
    ) -> pd.Series:
        n_seeds = self.cfg["lgbm_n_seeds"]
        X = sample[features].values.astype(np.float32)
        y = sample[target].values.astype(np.float32)

        all_imp: List[np.ndarray] = []

        with Progress(
            SpinnerColumn(), TextColumn("[info]LightGBM...[/info]"),
            BarColumn(bar_width=20),
            TextColumn("[highlight]Seed {task.completed}/{task.total}[/highlight]"),
            TimeElapsedColumn(), console=console, transient=True,
        ) as prog:
            task = prog.add_task("LGBM", total=n_seeds)
            for seed in range(n_seeds):
                self._print_ram(f"LGBM seed {seed}")
                model = LGBMRegressor(
                    n_estimators     = self.cfg["lgbm_n_estimators"],
                    importance_type  = "gain",
                    learning_rate    = self.cfg["lgbm_learning_rate"],
                    num_leaves       = self.cfg["lgbm_num_leaves"],
                    subsample        = self.cfg["lgbm_subsample"],
                    colsample_bytree = self.cfg["lgbm_colsample_bytree"],
                    min_child_samples= self.cfg["lgbm_min_child_samples"],
                    n_jobs           = self.cfg["lgbm_n_jobs"],
                    random_state     = seed,
                    verbose          = -1,
                )
                model.fit(X, y)
                all_imp.append(model.feature_importances_.copy())
                del model
                gc.collect()
                prog.advance(task)

        del X, y
        gc.collect()

        avg = np.mean(all_imp, axis=0)
        avg = avg / (avg.sum() + 1e-9)
        return pd.Series(avg, index=features).sort_values(ascending=False)

    # ──────────────────────────────────────────────────────────────────────────
    # 🏆 ENSEMBLE
    # ──────────────────────────────────────────────────────────────────────────

    def _compute_ensemble_score(
        self,
        features      : List[str],
        spearman_dict : Dict[str, float],
        mi_dict       : Dict[str, float],
        lgbm_imp      : pd.Series,
    ) -> pd.Series:
        w_sp   = self.cfg["spearman_weight"]
        w_mi   = self.cfg["mi_weight"]
        w_lgbm = self.cfg["lgbm_weight"]

        n = len(features)
        if n == 0:
            return pd.Series(dtype=float)

        df_scores = pd.DataFrame(index=features)
        df_scores["spearman"] = [spearman_dict.get(f, 0.0) for f in features]
        df_scores["mi"]       = [mi_dict.get(f, 0.0)       for f in features]
        df_scores["lgbm"]     = [lgbm_imp.get(f, 0.0)      for f in features]

        for col in ["spearman", "mi", "lgbm"]:
            ranks             = df_scores[col].rank(method="average")
            df_scores[col]    = (ranks - 1) / (n - 1) if n > 1 else ranks

        df_scores["ensemble"] = (
            w_sp   * df_scores["spearman"] +
            w_mi   * df_scores["mi"]       +
            w_lgbm * df_scores["lgbm"]
        )

        return df_scores["ensemble"].sort_values(ascending=False)

    # ──────────────────────────────────────────────────────────────────────────
    # 📊 FIGURAS
    # ──────────────────────────────────────────────────────────────────────────

    def _save_figures(self, result: SelectionResult, fig_dir: Path) -> None:
        fig_dir.mkdir(parents=True, exist_ok=True)
        top_n   = self.cfg["fig_top_n_importance"]
        dpi     = self.cfg["fig_dpi"]
        strat   = result.strategy
        cenario = result.cenario

        # ── 1. Ensemble Score Top-N ───────────────────────────────────────────
        try:
            top_ens = result.ensemble_score.head(top_n)
            fig, ax = plt.subplots(figsize=(13, max(6, top_n * 0.36)))
            colors  = [
                "#f39c12" if f in result.mandatory else
                "#2ecc71" if f in result.final_features else
                "#bdc3c7"
                for f in top_ens.index
            ]
            ax.barh(range(len(top_ens)), top_ens.values[::-1], color=colors[::-1])
            ax.set_yticks(range(len(top_ens)))
            ax.set_yticklabels(top_ens.index[::-1], fontsize=8)
            ax.set_xlabel("Score Ensemble (Spearman + MI + LightGBM, ranks norm.)", fontsize=10)
            ax.set_title(
                f"Top-{top_n} Features — Score Ensemble\n"
                f"{cenario} | Estratégia: {strat} | TARGET: {result.target_col}",
                fontsize=11,
            )
            from matplotlib.patches import Patch
            ax.legend(handles=[
                Patch(color="#f39c12", label="Obrigatória"),
                Patch(color="#2ecc71", label="Selecionada (ranking)"),
                Patch(color="#bdc3c7", label="Não selecionada"),
            ], loc="lower right", fontsize=8)
            plt.tight_layout()
            out = fig_dir / f"ensemble_score_top{top_n}_{strat}.png"
            fig.savefig(out, dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            console.print(f"   [success]🖼  {out.name}[/]")
        except Exception as e:
            console.print(f"   [warning]⚠  Figura ensemble falhou: {e}[/]")

        # ── 2. Comparação dos 3 métodos ───────────────────────────────────────
        try:
            feats_plot = result.ensemble_score.head(60).index.tolist()
            sp_vals  = [result.spearman_dict.get(f, 0.0) for f in feats_plot]
            mi_vals  = [result.mi_dict.get(f, 0.0)       for f in feats_plot]
            lgbm_v   = [float(result.lgbm_importance.get(f, 0.0)) for f in feats_plot]
            ens_vals = [float(result.ensemble_score.get(f, 0.0))   for f in feats_plot]

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            pairs = [
                (sp_vals,  ens_vals, "Spearman |r|",      "#3498db"),
                (mi_vals,  ens_vals, "Mutual Information", "#9b59b6"),
                (lgbm_v,   ens_vals, "LightGBM gain",     "#e74c3c"),
            ]
            for ax, (x, y, xlabel, color) in zip(axes, pairs):
                ax.scatter(x, y, alpha=0.6, s=18, color=color)
                ax.set_xlabel(xlabel, fontsize=9)
                ax.set_ylabel("Score Ensemble", fontsize=9)
                ax.set_title(f"{xlabel} vs Ensemble", fontsize=10)
                ax.grid(True, linestyle="--", alpha=0.3)
            fig.suptitle(
                f"Correlação entre métodos — {cenario} | {strat}", fontsize=11
            )
            plt.tight_layout()
            out = fig_dir / f"metodos_comparacao_{strat}.png"
            fig.savefig(out, dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            console.print(f"   [success]🖼  {out.name}[/]")
        except Exception as e:
            console.print(f"   [warning]⚠  Figura comparação falhou: {e}[/]")

        # ── 3. Heatmap Spearman ───────────────────────────────────────────────
        try:
            top_n_heat = self.cfg["fig_heatmap_top_n"]
            feats_heat = [f for f in result.final_features[:top_n_heat]
                          if f in result.spearman_dict]
            sp_sorted  = pd.Series({f: result.spearman_dict[f] for f in feats_heat}
                                    ).sort_values(ascending=False)

            fig, ax = plt.subplots(figsize=(2, max(4, len(sp_sorted) * 0.36)))
            im = ax.imshow(sp_sorted.values.reshape(-1, 1), aspect="auto",
                           cmap="RdYlGn", vmin=0, vmax=1)
            ax.set_yticks(range(len(sp_sorted)))
            ax.set_yticklabels(sp_sorted.index, fontsize=7)
            ax.set_xticks([])
            ax.set_title(f"Spearman |r| vs TARGET\n{strat}", fontsize=9)
            for idx, val in enumerate(sp_sorted.values):
                ax.text(0, idx, f"{val:.3f}", ha="center", va="center",
                        fontsize=6.5, color="black")
            plt.colorbar(im, ax=ax, fraction=0.15, pad=0.04)
            plt.tight_layout()
            out = fig_dir / f"spearman_heatmap_{strat}.png"
            fig.savefig(out, dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            console.print(f"   [success]🖼  {out.name}[/]")
        except Exception as e:
            console.print(f"   [warning]⚠  Figura heatmap falhou: {e}[/]")

        # ── 4. Distribuição do score ensemble ─────────────────────────────────
        try:
            fig, ax = plt.subplots(figsize=(8, 4))
            vals = result.ensemble_score.values
            ax.hist(vals, bins=40, color="#8e44ad", edgecolor="white", alpha=0.85)
            if len(result.final_features) > 0:
                min_sel_score = result.ensemble_score.reindex(result.final_features).dropna().min()
                ax.axvline(min_sel_score, color="#e74c3c", linestyle="--", linewidth=1.4,
                           label=f"Mínimo das selecionadas ({min_sel_score:.3f})")
            ax.set_xlabel("Score Ensemble", fontsize=10)
            ax.set_ylabel("Frequência", fontsize=10)
            ax.set_title(
                f"Distribuição do Score Ensemble — {strat}\n"
                f"{cenario} | {len(vals)} features avaliadas",
                fontsize=11,
            )
            ax.legend(fontsize=9)
            plt.tight_layout()
            out = fig_dir / f"ensemble_distribution_{strat}.png"
            fig.savefig(out, dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            console.print(f"   [success]🖼  {out.name}[/]")
        except Exception as e:
            console.print(f"   [warning]⚠  Figura distribuição falhou: {e}[/]")

    # ──────────────────────────────────────────────────────────────────────────
    # 💾 EXPORTAÇÃO
    # ──────────────────────────────────────────────────────────────────────────

    def _export(
        self,
        result   : SelectionResult,
        df_treino: pd.DataFrame,
        df_teste : pd.DataFrame,
        out_dir  : Path,
    ) -> None:
        out_dir.mkdir(parents=True, exist_ok=True)
        fig_dir = out_dir / "figuras"

        target_col  = result.target_col
        cols_treino = [c for c in result.final_features if c in df_treino.columns]
        if target_col not in cols_treino:
            cols_treino.append(target_col)
        cols_teste = [c for c in cols_treino if c in df_teste.columns]

        with self.timer.track(f"[{result.cenario}/{result.strategy}] Salvar TREINO_SELECTED"):
            df_treino[cols_treino].astype(np.float32).to_parquet(
                out_dir / "TREINO_SELECTED.parquet", index=False
            )
        console.print(f"   [success]✅ TREINO_SELECTED salvo ({len(cols_treino)} colunas)[/]")

        with self.timer.track(f"[{result.cenario}/{result.strategy}] Salvar TESTE_SELECTED"):
            df_teste[cols_teste].astype(np.float32).to_parquet(
                out_dir / "TESTE_SELECTED.parquet", index=False
            )
        console.print(f"   [success]✅ TESTE_SELECTED salvo ({len(cols_teste)} colunas)[/]")

        # ── CSV de ranking ────────────────────────────────────────────────────
        rows = []
        for rank, feat in enumerate(result.ensemble_score.index, start=1):
            rows.append({
                "rank"              : rank,
                "feature"           : feat,
                "tipo"              : "OBRIGATORIA" if feat in result.mandatory else "RANKING",
                "selecionada"       : "SIM" if feat in result.final_features else "NAO",
                "score_ensemble"    : round(float(result.ensemble_score.get(feat, 0)), 8),
                "spearman_abs"      : round(result.spearman_dict.get(feat, float("nan")), 6),
                "mutual_information": round(result.mi_dict.get(feat, float("nan")), 6),
                "lgbm_gain_norm"    : round(float(result.lgbm_importance.get(feat, 0)), 8),
            })
        pd.DataFrame(rows).to_csv(
            out_dir / f"RANKING_{result.strategy}.csv",
            index=False, sep=";", decimal=",", encoding="utf-8-sig",
        )
        console.print(f"   [success]✅ RANKING CSV salvo[/]")

        # ── TXT legível ───────────────────────────────────────────────────────
        with open(out_dir / f"FEATURES_{result.strategy}.txt", "w", encoding="utf-8") as fh:
            fh.write(f"# ═══════════════════════════════════════════════════════════\n")
            fh.write(f"# Seleção de Variáveis — TCC PLD Nordeste\n")
            fh.write(f"# Cenário    : {result.cenario}\n")
            fh.write(f"# Estratégia : {result.strategy}\n")
            fh.write(f"# Target     : {result.target_col}\n")
            fh.write(f"# Gerado em  : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n")
            fh.write(f"# Método     : Ensemble (Spearman {self.cfg['spearman_weight']*100:.0f}% "
                     f"+ MI {self.cfg['mi_weight']*100:.0f}% "
                     f"+ LightGBM {self.cfg['lgbm_weight']*100:.0f}%)\n")
            fh.write(f"# Total      : {len(result.final_features)} features selecionadas\n")
            fh.write(f"# ═══════════════════════════════════════════════════════════\n\n")
            fh.write("# ── OBRIGATÓRIAS ─────────────────────────────────────────\n")
            for f in result.mandatory:
                fh.write(f"{f}\n")
            fh.write("\n# ── POR RANKING ENSEMBLE (ordem decrescente de score) ────\n")
            fh.write(f"# {'#':>3}  {'Feature':<55}  {'Ensemble':>9}  "
                     f"{'Spearman':>9}  {'MI':>9}  {'LGBM':>9}\n")
            for i, f in enumerate(result.top_additional, start=1):
                ens  = float(result.ensemble_score.get(f, 0.0))
                sp   = result.spearman_dict.get(f, float("nan"))
                mi   = result.mi_dict.get(f, float("nan"))
                lgbm = float(result.lgbm_importance.get(f, 0.0))
                fh.write(
                    f"{i:>3}. {f:<55}  {ens:>9.6f}  {sp:>9.4f}  {mi:>9.4f}  {lgbm:>9.6f}\n"
                )
        console.print(f"   [success]✅ FEATURES TXT salvo[/]")

        # ── JSON de parâmetros ────────────────────────────────────────────────
        params_json = {
            "cenario"               : result.cenario,
            "strategy"              : result.strategy,
            "target_col"            : result.target_col,
            "gerado_em"             : datetime.now().isoformat(),
            "metodo"                : "ensemble_spearman_mi_lgbm",
            "pesos"                 : {
                "spearman" : self.cfg["spearman_weight"],
                "mi"       : self.cfg["mi_weight"],
                "lgbm"     : self.cfg["lgbm_weight"],
            },
            "n_features_final"      : len(result.final_features),
            "n_treino_rows"         : result.n_treino_rows,
            "n_teste_rows"          : result.n_teste_rows,
            "n_candidates_initial"  : result.n_candidates_initial,
            "n_after_variance"      : result.n_after_variance,
            "n_after_multicol"      : result.n_after_multicol,
            "pipeline_config"       : self.cfg,
            "mandatory_features"    : result.mandatory,
            "selected_features"     : result.final_features,
        }
        with open(out_dir / "feature_params_selection.json", "w", encoding="utf-8") as fh:
            json.dump(params_json, fh, ensure_ascii=False, indent=2)
        console.print(f"   [success]✅ JSON de parâmetros salvo[/]")

        # ── Figuras ───────────────────────────────────────────────────────────
        console.print(f"\n   [info]🖼  Gerando figuras em {fig_dir}...[/]")
        self._save_figures(result, fig_dir)

    # ──────────────────────────────────────────────────────────────────────────
    # 📊 DISPLAY DE RANKING
    # ──────────────────────────────────────────────────────────────────────────

    def _print_ranking_table(self, result: SelectionResult) -> None:
        console.print()
        console.print(Panel(
            f"[bold white]📊 RANKING ENSEMBLE — {result.cenario} | "
            f"[cyan]{result.strategy}[/cyan] | TARGET: [cyan]{result.target_col}[/cyan][/bold white]",
            border_style="cyan", padding=(0, 2),
        ))

        t = Table(
            box=box.ROUNDED, show_header=True,
            header_style="bold white on dark_blue",
            show_lines=False, padding=(0, 1),
        )
        t.add_column("#",        style="muted",  justify="right",  width=4)
        t.add_column("Feature",  style="bold",   justify="left",   min_width=38)
        t.add_column("Tipo",     justify="center", width=11)
        t.add_column("Ensemble", justify="right",  width=9)
        t.add_column("Spearman", justify="right",  width=9)
        t.add_column("MI",       justify="right",  width=9)
        t.add_column("LightGBM", justify="right",  width=9)
        t.add_column("Status",   justify="center", width=11)

        full_list = list(result.ensemble_score.index)

        for rank, feat in enumerate(full_list, start=1):
            is_mandatory = feat in result.mandatory
            is_selected  = feat in result.final_features
            ens_val      = float(result.ensemble_score.get(feat, 0.0))
            sp_val       = result.spearman_dict.get(feat, float("nan"))
            mi_val       = result.mi_dict.get(feat, float("nan"))
            lgbm_val     = float(result.lgbm_importance.get(feat, 0.0))
            rank_style   = ("rank_top" if rank <= 10 else
                            "rank_mid" if rank <= 30 else "rank_low")
            tipo_str     = "[mandatory]OBRIG.[/mandatory]" if is_mandatory else "[muted]ranking[/muted]"
            status_str   = "[selected]✔ ATIVA[/selected]" if is_selected else "[rejected]✘ skip[/rejected]"
            sp_str       = f"{sp_val:.4f}" if not np.isnan(sp_val) else "  —  "
            mi_str       = f"{mi_val:.4f}" if not np.isnan(mi_val) else "  —  "

            t.add_row(
                f"[{rank_style}]{rank:>3}[/{rank_style}]",
                feat, tipo_str,
                f"{ens_val:.5f}", sp_str, mi_str, f"{lgbm_val:.6f}",
                status_str,
            )
        console.print(t)

    def _print_stage_summary(self, result: SelectionResult) -> None:
        t = Table(
            box=box.SIMPLE_HEAVY, show_header=True, header_style="info", padding=(0, 2)
        )
        t.add_column("Etapa",    style="muted",   width=38)
        t.add_column("Features", style="success", justify="right", width=12)
        t.add_row("Candidatas iniciais",            str(result.n_candidates_initial))
        t.add_row("Após filtro variância zero",     str(result.n_after_variance))
        t.add_row("Após filtro multicolinearidade", str(result.n_after_multicol))
        t.add_row("Avaliadas pelo ensemble",        str(len(result.ensemble_score)))
        t.add_row("[mandatory]Obrigatórias (cota)[/mandatory]",
                  f"[mandatory]{len(result.mandatory)}[/mandatory]")
        t.add_row("[selected]Por ranking ensemble[/selected]",
                  f"[selected]{len(result.top_additional)}[/selected]")
        t.add_row("[bold]TOTAL SELECIONADAS[/bold]",
                  f"[bold]{len(result.final_features)}[/bold]")
        console.print(Panel(
            t,
            title=f"[header] 📋 RESUMO — {result.cenario} | {result.strategy} [/header]",
            border_style="green", padding=(0, 2),
        ))

    # ──────────────────────────────────────────────────────────────────────────
    # 🚀 PIPELINE POR ESTRATÉGIA
    # ──────────────────────────────────────────────────────────────────────────

    def _run_strategy(self, cenario_label: str, strategy: str) -> Optional[Dict]:
        console.print()
        console.rule(f"[header] {cenario_label} | ESTRATÉGIA: {strategy} [/header]")

        treino_in = (self.input_dir / cenario_label / "treino" /
                     strategy / "TREINO_FEATURES.parquet")
        teste_in  = (self.input_dir / cenario_label / "teste" /
                     strategy / "TESTE_FEATURES.parquet")
        out_dir   = self.save_dir / cenario_label / strategy

        if not treino_in.exists():
            console.print(f"   [error]✘  TREINO não encontrado: {treino_in}[/]")
            return {
                "cenario": cenario_label, "strategy": strategy,
                "status" : "ERRO (arquivo não encontrado)",
            }

        t0 = time.perf_counter()

        # ── Leitura ───────────────────────────────────────────────────────────
        with self.timer.track(f"[{cenario_label}/{strategy}] Leitura TREINO"):
            df_treino = self._load_split(treino_in, "TREINO")
        n_treino = len(df_treino)

        df_teste = None
        n_teste  = 0
        if teste_in.exists():
            with self.timer.track(f"[{cenario_label}/{strategy}] Leitura TESTE"):
                df_teste = self._load_split(teste_in, "TESTE")
            n_teste = len(df_teste)
        else:
            console.print(f"   [warning]⚠  TESTE não encontrado: {teste_in}[/]")

        # ── Target ────────────────────────────────────────────────────────────
        console.print("\n   [info]🎯 Detectando TARGET...[/]")
        target_col   = detectar_target(df_treino)
        all_cols_set = set(df_treino.columns)

        exclude_set = {target_col} | {c for c in all_cols_set if _is_excluded(c)}

        # ── Obrigatórias disponíveis ──────────────────────────────────────────
        mandatory = [c for c in FEATURES_OBRIGATORIAS
                     if c in all_cols_set and c not in exclude_set]
        if len(mandatory) < len(FEATURES_OBRIGATORIAS):
            ausentes = [c for c in FEATURES_OBRIGATORIAS if c not in all_cols_set]
            if ausentes:
                console.print(
                    f"   [warning]⚠  {len(ausentes)} obrigatórias ausentes no parquet: "
                    f"{ausentes[:5]}{'…' if len(ausentes) > 5 else ''}[/]"
                )

        # ── Candidatas ────────────────────────────────────────────────────────
        candidates = [
            c for c in df_treino.select_dtypes(include=[np.number]).columns
            if c not in exclude_set and c not in set(mandatory)
        ]
        n_initial = len(candidates)
        console.print(
            f"   [info]🔢 Candidatas: {n_initial} | Obrigatórias: {len(mandatory)}[/]"
        )
        self._print_ram("início")

        # ── Amostra do TREINO (anti-leakage) ──────────────────────────────────
        n_sample = min(n_treino, self.cfg["sample_size"])
        sample   = df_treino.sample(n_sample, random_state=42).copy()
        console.print(f"   [info]📊 Amostra de fit: {n_sample:,} linhas do TREINO[/]")

        # ── PRÉ-FILTRO 1: Variância zero ──────────────────────────────────────
        with self.timer.track(f"[{cenario_label}/{strategy}] Variância zero"):
            candidates, n_rem_var = self._remove_zero_variance(candidates, sample)
        n_after_var = len(candidates)
        console.print(
            f"   [info]⚙️  Variância zero:[/] "
            f"[error]-{n_rem_var}[/] → [success]{n_after_var}[/] restantes"
        )

        # ── PRÉ-FILTRO 2: Multicolinearidade ─────────────────────────────────
        with self.timer.track(f"[{cenario_label}/{strategy}] Multicolinearidade"):
            if len(candidates) > 30:
                sub = sample[candidates].sample(
                    min(len(sample), self.cfg["sub_sample_multicol"]),
                    random_state=42,
                )
                candidates, n_rem_mc = self._remove_multicolinearidade(candidates, sub)
                self._force_free(sub)
            else:
                n_rem_mc = 0
        n_after_mc = len(candidates)
        console.print(
            f"   [info]🔗 Multicolinearidade:[/] "
            f"[error]-{n_rem_mc}[/] → [success]{n_after_mc}[/] restantes"
        )
        self._print_ram("pós-multicol")

        # ── MÉTODO A: Spearman ────────────────────────────────────────────────
        console.print(f"\n   [info]📐 Método A — Spearman ({n_after_mc} candidatas)...[/]")
        with self.timer.track(f"[{cenario_label}/{strategy}] Spearman"):
            spearman_dict = self._compute_spearman(candidates, sample, target_col)
        console.print(
            f"   [success]✔  Spearman concluído. "
            f"Máximo: {max(spearman_dict.values()):.4f}[/]"
        )
        self._print_ram("pós-Spearman")

        # ── MÉTODO B: Mutual Information ──────────────────────────────────────
        console.print(f"\n   [info]📐 Método B — Mutual Information ({n_after_mc} candidatas)...[/]")
        with self.timer.track(f"[{cenario_label}/{strategy}] Mutual Information"):
            mi_dict = self._compute_mi(candidates, sample, target_col)
        console.print(
            f"   [success]✔  MI concluída. "
            f"Máximo: {max(mi_dict.values()):.4f}[/]"
        )
        self._print_ram("pós-MI")

        # ── Pool para o LightGBM (top Spearman ∪ top MI ∪ obrigatórias) ──────
        top_n_pool = self.cfg["top_corr_candidates"]
        top_sp     = sorted(spearman_dict, key=spearman_dict.get, reverse=True)[:top_n_pool]
        top_mi     = sorted(mi_dict,       key=mi_dict.get,       reverse=True)[:top_n_pool]
        lgbm_pool  = list(dict.fromkeys(mandatory + top_sp + top_mi))
        console.print(
            f"\n   [ensemble]🔀 Pool LGBM: {len(lgbm_pool)} features "
            f"(top-{top_n_pool} Spearman ∪ top-{top_n_pool} MI ∪ obrigatórias)[/]"
        )

        # ── MÉTODO C: LightGBM ────────────────────────────────────────────────
        console.print(
            f"\n   [info]🌲 Método C — LightGBM ({len(lgbm_pool)} features · "
            f"{self.cfg['lgbm_n_seeds']} seeds)...[/]"
        )
        with self.timer.track(f"[{cenario_label}/{strategy}] LightGBM"):
            lgbm_imp = self._compute_lgbm(lgbm_pool, sample, target_col)
        self._print_ram("pós-LightGBM")

        # ── ENSEMBLE ──────────────────────────────────────────────────────────
        all_evaluated = list(dict.fromkeys(candidates + mandatory))
        console.print(
            f"\n   [ensemble]🏆 Calculando score ensemble "
            f"({len(all_evaluated)} features)...[/]"
        )
        ensemble_score = self._compute_ensemble_score(
            all_evaluated, spearman_dict, mi_dict, lgbm_imp
        )

        # ── Seleção final ─────────────────────────────────────────────────────
        ranking_only   = ensemble_score.drop(labels=mandatory, errors="ignore")
        n_needed       = self.cfg["n_total_features"] - len(mandatory)
        top_additional = ranking_only.head(max(0, n_needed)).index.tolist()
        final_features = mandatory + top_additional

        result = SelectionResult(
            cenario              = cenario_label,
            strategy             = strategy,
            target_col           = target_col,
            mandatory            = mandatory,
            top_additional       = top_additional,
            final_features       = final_features,
            spearman_dict        = spearman_dict,
            mi_dict              = mi_dict,
            lgbm_importance      = lgbm_imp,
            ensemble_score       = ensemble_score,
            n_candidates_initial = n_initial,
            n_after_variance     = n_after_var,
            n_after_multicol     = n_after_mc,
            n_treino_rows        = n_treino,
            n_teste_rows         = n_teste,
        )

        self._print_stage_summary(result)
        self._print_ranking_table(result)

        # ── Exportação ────────────────────────────────────────────────────────
        console.print(Rule("[info]💾  EXPORTAÇÃO[/info]"))
        if df_teste is None:
            df_teste = pd.DataFrame(columns=df_treino.columns)

        with self.timer.track(f"[{cenario_label}/{strategy}] Exportação"):
            self._export(result, df_treino, df_teste, out_dir)

        self._force_free(df_treino, df_teste, sample, lgbm_pool, top_sp, top_mi)

        elapsed = time.perf_counter() - t0
        self._print_ram("após liberação")

        return {
            "cenario"          : cenario_label,
            "strategy"         : strategy,
            "target_col"       : target_col,
            "n_features_final" : len(final_features),
            "n_mandatory"      : len(mandatory),
            "n_ranking"        : len(top_additional),
            "n_treino_rows"    : n_treino,
            "n_teste_rows"     : n_teste,
            "elapsed_s"        : round(elapsed, 2),
            "status"           : "OK",
        }

    # ──────────────────────────────────────────────────────────────────────────
    # 📋 RELATÓRIOS FINAIS
    # ──────────────────────────────────────────────────────────────────────────

    def _print_time_report(self, elapsed_total: float) -> None:
        console.print()
        console.print(Rule("[success]⏱️  RELATÓRIO DE TEMPO[/success]"))
        t = Table(box=box.SIMPLE_HEAVY, show_header=True, header_style="info")
        t.add_column("Estágio",   style="muted",   min_width=50)
        t.add_column("Tempo (s)", style="success", justify="right", width=12)
        for label, elapsed in self.timer.records.items():
            t.add_row(label, f"{elapsed:.2f}s")
        t.add_row("[bold]TOTAL[/bold]", f"[bold]{elapsed_total:.2f}s[/bold]")
        console.print(t)

    def _print_final_table(self) -> None:
        t = Table(
            title="RELATÓRIO CONSOLIDADO — SELEÇÃO DE VARIÁVEIS | PLD NORDESTE",
            border_style="blue", box=box.DOUBLE_EDGE, show_lines=True, min_width=120,
        )
        t.add_column("Cenário",    style="bold",      justify="left",  min_width=28)
        t.add_column("Estratégia", style="bold cyan", justify="left",  min_width=20)
        t.add_column("TARGET",     style="dim",       justify="left",  min_width=30)
        t.add_column("Features",   style="green",     justify="right", min_width=9)
        t.add_column("Obrigat.",   style="yellow",    justify="right", min_width=9)
        t.add_column("Ranking",    style="cyan",      justify="right", min_width=8)
        t.add_column("Treino",     justify="right",   min_width=12)
        t.add_column("Teste",      justify="right",   min_width=10)
        t.add_column("Tempo (s)",  style="white",     justify="right", min_width=9)
        t.add_column("Status",     style="bold",      justify="center", min_width=8)
        for r in self._all_results:
            is_a   = "todos" in r.get("cenario", "")
            cor_c  = "cenario_a" if is_a else "cenario_b"
            status = "[green]OK[/]" if r.get("status") == "OK" else f"[red]{r.get('status')}[/]"
            t.add_row(
                f"[{cor_c}]{r.get('cenario', '—')}[/{cor_c}]",
                r.get("strategy", "—"),
                r.get("target_col", "—"),
                str(r.get("n_features_final", 0)),
                str(r.get("n_mandatory", 0)),
                str(r.get("n_ranking", 0)),
                f"{r.get('n_treino_rows', 0):,}",
                f"{r.get('n_teste_rows', 0):,}",
                str(r.get("elapsed_s", 0)),
                status,
            )
        console.print(t)

    def _save_consolidated_report(self) -> None:
        out = self.save_dir / "relatorio_selecao_variaveis_nordeste.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(self._all_results).to_csv(out, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório consolidado: {out}[/]")

    # ──────────────────────────────────────────────────────────────────────────
    # 🚀 PONTO DE ENTRADA
    # ──────────────────────────────────────────────────────────────────────────

    def run(self) -> None:
        console.print()
        console.print(Panel(
            Text.assemble(
                ("🔬 SELEÇÃO DE VARIÁVEIS — PLD NORDESTE\n", "bold green"),
                ("   Cenário A (todos os anos)\n", "muted"),
                (f"   Input  : {self.input_dir}\n", "muted"),
                (f"   Output : {self.save_dir}\n", "muted"),
                (f"   Run em : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n", "muted"),
                ("   Modo   : ", "muted"),
                ("🏆 Ensemble Spearman + MI + LightGBM | Memory-Safe | Anti-Leakage", "warning"),
            ),
            title="[header]  TCC PLD — STEP 9: SELEÇÃO DE VARIÁVEIS | NORDESTE  [/header]",
            border_style="green", padding=(1, 4),
        ))

        console.print(Panel(
            "[warning]GARANTIA ANTI-LEAKAGE:[/]\n\n"
            "  Variância, multicolinearidade, Spearman, MI e LightGBM\n"
            "  são calculados SOMENTE na amostra do TREINO.\n\n"
            "  O TESTE nunca influencia nenhuma decisão de seleção.\n"
            "  Apenas as colunas selecionadas são aplicadas ao TESTE.\n\n"
            "[ensemble]  MÉTODO ENSEMBLE:[/]\n"
            f"  Spearman {self.cfg['spearman_weight']*100:.0f}% + "
            f"MI {self.cfg['mi_weight']*100:.0f}% + "
            f"LightGBM {self.cfg['lgbm_weight']*100:.0f}%\n"
            "  Score = média ponderada de ranks normalizados [0,1]\n"
            "  → Justo para relações lineares E não-lineares",
            title="[header] ARQUITETURA [/header]",
            border_style="yellow", padding=(0, 2),
        ))

        t_total = time.perf_counter()

        for scenario in self.scenarios:
            cenario_label = scenario["label"]
            cor           = scenario["cor"]
            # Mapeia "cenario_a"/"cenario_b" → cor Rich válida
            border_color  = _BORDER_STYLE_MAP.get(cor, "white")

            console.print()
            console.rule(
                f"[{cor}]{'═'*20}  CENÁRIO: {cenario_label}  {'═'*20}[/{cor}]"
            )
            console.print(Panel(
                f"[{cor}]{scenario['desc']}[/{cor}]\n"
                f"[muted]Input : {self.input_dir / cenario_label}[/]\n"
                f"[muted]Output: {self.save_dir  / cenario_label}[/]",
                border_style=border_color,
                padding=(0, 2),
            ))

            for strategy in self.strategies:
                row = self._run_strategy(cenario_label, strategy)
                if row:
                    self._all_results.append(row)

        elapsed_total = time.perf_counter() - t_total
        self._print_time_report(elapsed_total)
        self._print_final_table()
        self._save_consolidated_report()

        n_ok  = sum(1 for r in self._all_results if r.get("status") == "OK")
        n_err = len(self._all_results) - n_ok
        used, tot, style = self._ram_status()

        console.print()
        console.print(Panel(
            Text.assemble(
                ("✅ PIPELINE NORDESTE CONCLUÍDO\n", "bold green"),
                (f"   {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n", "muted"),
                (f"   Estratégias OK : {n_ok} | Erros: {n_err}\n", "muted"),
                (f"   Tempo total    : {elapsed_total:.2f}s\n", "muted"),
                (f"   RAM final      : {used:.1f}/{tot:.1f} GB", style),
            ),
            border_style="green", padding=(1, 4),
        ))


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    engine = FeatureSelectionEngine(
        input_dir  = INPUT_DIR,
        save_dir   = SAVE_DIR,
        scenarios  = SCENARIOS,
        strategies = STRATEGIES,
        config     = PIPELINE_CONFIG,
    )
    engine.run()